In [ ]:
"""
intput: video file
output:
#  Data Adapter ：
pose_data_payload = {
    0: { # Frame 0
        "right_shoulder": [x,y,z],
        "right_elbow": [x,y,z],
        "right_wrist": [x,y,z],
        "right_hip": [x,y,z]
    },
    1: { # Frame 1
        "right_shoulder": [0.52, 0.50, 0.50],
        "right_elbow": [0.65, 0.20, 0.55],
        "right_wrist": [0.75, 0.05, 0.60],
        "right_hip": [0.51, 0.80, 0.50]
    },
    # ...
}
"""
from typing import Any
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np


def round_landmark(landmark):
    return [round(landmark.x, 3), round(landmark.y, 3), round(landmark.z, 3)]


model_path = "../models/pose_landmarker_heavy.task"
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    min_pose_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False,
)
# Create a PoseLandmarker object
detector = vision.PoseLandmarker.create_from_options(options)

# init the pose data payload
pose_data_payload: dict[int, Any] = {}
frame_idx = 0
# Open the webcam
cap = cv2.VideoCapture("../cache/raw_videos/test_clear_trim3.mp4")
fps = round(cap.get(cv2.CAP_PROP_FPS), 3)
print(f"Frames per second: {fps}")
while True:
    # Process the video frames
    success, frame = cap.read()
    if not success:
        print("End of video reached or failed to read the video frame.")
        break

    # Convert the BGR image to RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    # set up for timestamp in milliseconds for the current frame
    timestamp_ms = int((frame_idx / fps) * 1000)

    detection_result = detector.detect_for_video(mp_image, timestamp_ms=timestamp_ms)
    # ensure the detection result contains pose landmarks
    if detection_result.pose_landmarks:
        print("Pose landmarks detected:")

        # Extract all the 33 points
        # Note: detection_result.pose_landmarks is a list of PoseLandmarkList, where each PoseLandmarkList corresponds to a detected person in the frame. For simplicity, we will only consider the first detected person (if multiple people are detected).
        landmarks = detection_result.pose_landmarks[0]

        target_indices = [11, 12, 15, 16, 23, 24, 25, 26, 27, 28]

        frame_coords = [round_landmark(landmarks[i]) for i in target_indices]
        frame_matrix = np.array(frame_coords)
        print(frame_matrix)
        left_hip = frame_matrix[4]
        right_hip = [5]

        midpoint = frame_matrix[[4, 5]].mean(axis=0)
        normalized_coords = frame_matrix - midpoint

        pose_data_payload[frame_idx] = normalized_coords
    else:
        pose_data_payload[frame_idx] = np.zeros(
            (10, 3)
        )  # Use a list of zeros for frames with no detected landmarks
        print("No pose landmarks detected.")
    frame_idx += 1
print("pose_data_payload:", pose_data_payload)
cap.release()
detector.close()


In [ ]:
import numpy as np


video_tensor = np.array(list(pose_data_payload.values()))  # Convert the pose data payload to a NumPy array
# print("video_tensor shape:", video_tensor)
total_frames = video_tensor.shape[0]
print("Total frames processed:", total_frames)
print("video_tensor shape:", video_tensor.shape)

video_tensor_flat = video_tensor.reshape(total_frames, -1)  # Flatten the last two dimensions
print("video_tensor_flat shape:", video_tensor_flat.shape)
print("video_tensor_flat:", video_tensor_flat)
SQE_LEN = 11
window = []
for i in range(total_frames - SQE_LEN + 1):
    window.append(video_tensor_flat[i : i + SQE_LEN])

final_input_sensor = np.array(window)
print("final_input_sensor shape:", final_input_sensor.shape)
print("final_input_sensor:", final_input_sensor)

In [ ]:
import torch
import torch.nn as nn

class BadmintonMovementLSTM(nn.Module):
    def __init__(self):
        super(BadmintonMovementLSTM, self).__init__()

        self.lstm = nn.LSTM(
            input_size=30,  # 10 landmarks * 3 coordinates each
            hidden_size=64,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(
            in_features=64,
            out_features=1
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        """x shape: (batch_size, sequence_length=11, input_size=30)"""
        lstm_out, _ = self.lstm(x) # ht=LSTM(xt,ht-1)

        last_time_step_out = lstm_out[:, -1, :] # Get the output of the last time step

        dropout_out = self.dropout(last_time_step_out)

        logits = self.fc(dropout_out)

        probability = self.sigmoid(logits)

        return probability



model = BadmintonMovementLSTM()

# print(model)
dummy_input = torch.tensor(final_input_sensor, dtype=torch.float32)  # Convert the final input sensor to a PyTorch tensor

print("dummy_input shape:", dummy_input.shape)
prediction = model(dummy_input)
print("prediction shape:", prediction.shape)
print("prediction:", prediction[:5])  # Print the first 5 predictions

In [ ]:
import torch.optim as optim

